# Using Skills to Extend the Agent Capabilities
In this notebook, we will learn how to extend the agent’s context by providing additional information—such as business logic, data mappings, or domain-specific rules—that cannot be reliably inferred from the data alone.


<figure>
 <img src="../assets/chapter_2.png" width="60%" align="center"/></a>
<figcaption> Prompt Template Architecture </figcaption>
</figure>

<br>
<br />

## Setting the Database Connection

The below code enables us to connect to Postgres (or DuckDB) using the `get_ibis_connection` function:

In [2]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection



Setting a connection to the Postgres database:

In [3]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "localhost",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

Or, setting a connection to the DuckDB database:

In [4]:
# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )

LLM settings:

In [5]:
base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

## Loading and Injecting Skills

In [6]:
from sql_ai_agent.skill_manager import SkillManager

In [7]:
# Initialize skill manager
skill_manager = SkillManager(skills_dir= "../skills")

# List available skills
print("Available skills:")
for skill in skill_manager.list_skills():
    print(f"  - {skill}")



Available skills:
  - sfo_air_traffic_context


In [8]:
# Load the SFO air traffic skill
sfo_skill = skill_manager.load_skill("sfo_air_traffic_context")
print(f"\n✓ Loaded skill: {len(sfo_skill):,} characters")



✓ Loaded skill: 1,501 characters


In [9]:
print(sfo_skill)

# SFO Air Traffic Passenger Statistics - Domain Knowledge

## Schema Reference

Table: air_traffic, monthly passenger data from July 1999 onward.

- Year: 4-digit year.
- Date: first day of month.
- Operating Airline: airline that operated the flight.
- Operating Airline IATA Code: 2-letter code, may be null.
- Published Airline: marketing airline, relevant for codeshare tickets.
- Published Airline IATA Code: 2-letter code, may be null.
- GEO Summary: Domestic or International.
- GEO Region: US, Europe, Asia, Canada, Mexico, and similar regions.
- Activity Type Code: Deplaned, Enplaned, or Thru / Transit.
- Price Category Code: Low Fare or Other.
- Terminal: terminal name.
- Boarding Area: boarding area letter.
- Passenger Count: monthly passenger total.

Column names with spaces must be quoted, for example `"Operating Airline"` and
`"Passenger Count"`.

## Query Guidance

Most passenger analyses should exclude transit passengers:

```sql
WHERE "Activity Type Code" IN ('Deplaned', 'En

In [10]:

from sql_ai_agent.SqlAgent import SqlAgent

agent_with_skill = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    tbl_name=tbl_name,
    fallback=False,
    fallback_model=model,
    skill=True,
    skills_dir="../skills",
)

agent_without_skill = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    tbl_name=tbl_name,
    fallback=False,
    fallback_model=model,
    skill=False,
)

print("✓ Agent initialized")


✓ Agent initialized


In [11]:
complex_question = """
  Which international airlines showed the biggest growth from 2023 to 2024?
  Show the top 10 airlines ranked by percentage increase in passenger volume.
  For each airline, show their 2023 total, 2024 total, and calculate year-over-year growth

  Make sure to exclude transit passengers and avoid double-counting from code share agreements.
  """

In [12]:
result_without_skill = agent_without_skill.ask_question(
    question= complex_question,
    distinct_char_values = True,
    verbose=True,
    trials= 0
)

ℹ️  Query modified to enforce LIMIT: 10000

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  "Operating Airline",
  SUM(CASE WHEN EXTRACT(YEAR FROM "Date") = 2023 THEN "Passenger Count" ELSE 0 END) AS "2023 Total",
  SUM(CASE WHEN EXTRACT(YEAR FROM "Date") = 2024 THEN "Passenger Count" ELSE 0 END) AS "2024 Total",
  CASE
    WHEN SUM(CASE WHEN EXTRACT(YEAR FROM "Date") = 2023 THEN "Passenger Count" ELSE 0 END) = 0
    THEN NULL
    ELSE CAST((
      SUM(CASE WHEN EXTRACT(YEAR FROM "Date") = 2024 THEN "Passenger Count" ELSE 0 END) - SUM(CASE WHEN EXTRACT(YEAR FROM "Date") = 2023 THEN "Passenger Count" ELSE 0 END)
    ) AS FLOAT) / SUM(CASE WHEN EXTRACT(YEAR FROM "Date") = 2023 THEN "Passenger Count" ELSE 0 END) * 100
  END AS "Year-over-Year Growth (%)"
FROM air_traffic
WHERE
  "GEO Summary" = 'International' AND "Activity Type Code" <> 'Thru / Transit'
GROUP BY
  "Operating Airline"
ORDER BY
  "Year-over-Year Growth (%)" DESC
LIMIT 10

Results (10 rows):
----------------------------------------

In [13]:
result_with_skill = agent_with_skill.ask_question(
    question=complex_question,
    distinct_char_values = True,
    verbose=True,
    trials= 0
)


ℹ️  Query modified to enforce LIMIT: 10000

✓ QUERY SUCCESSFUL

SQL Query:
WITH yearly_passenger_totals AS (
  SELECT
    "Operating Airline",
    "Year",
    SUM("Passenger Count") AS total_passengers
  FROM air_traffic
  WHERE
    "Activity Type Code" IN ('Deplaned', 'Enplaned')
    AND "GEO Summary" = 'International'
    AND "Year" IN (2023, 2024)
  GROUP BY
    "Operating Airline",
    "Year"
), passenger_growth AS (
  SELECT
    a."Operating Airline",
    a.total_passengers AS total_2023,
    b.total_passengers AS total_2024,
    (
      (
        b.total_passengers - a.total_passengers
      ) * 100.0
    ) / NULLIF(a.total_passengers, 0) AS growth_percentage
  FROM yearly_passenger_totals AS a
  JOIN yearly_passenger_totals AS b
    ON a."Operating Airline" = b."Operating Airline"
    AND a."Year" = 2023
    AND b."Year" = 2024
)
SELECT
  "Operating Airline",
  total_2023,
  total_2024,
  growth_percentage
FROM passenger_growth
ORDER BY
  growth_percentage DESC
LIMIT 10

Results